# Dive.ai 웹사이트 UI & 기능 구현 명세

> 이 문서는 벡터DB.ipynb / 파이프라인.ipynb에서 설계·구현된 백엔드 로직을 기반으로,
> 웹사이트에 구현해야 할 **화면(UI)과 기능**을 정리한 것입니다.
> 이미 기본 디자인이 적용된 웹사이트에 아래 UI/기능을 추가해 주세요.

## 전체 화면 흐름

```
[화면 1] 진입 — 콘텐츠 유형 & 장르 선택
    ↓
[화면 2] 소재 & 캐릭터 입력
    ↓
[화면 3] 생성 중 로딩
    ↓
[화면 4] 완료 화면 — 시나리오 확인 + 캐릭터 확인/수정
    ↓
[화면 5] 플레이 시작 전 설정
    ↓
[화면 6] 인터랙티브 챗 & 플레이
    ↓
[화면 7] 엔딩 결과 화면
```

**서포트 레이어**: 로어북 (전 단계에 걸쳐 작동)
**계정**: 구글 로그인 / 세션 관리

## 화면 1 — 진입: 콘텐츠 유형 & 장르 선택

### UI 요소

| 요소 | 설명 |
|------|------|
| 콘텐츠 유형 선택 버튼 | 만화 / 시리즈 / 영화 / 소설 / 고전 (5개 버튼) |
| **무작위** 버튼 | 콘텐츠 유형과 장르 선택에 무작위 버튼을 만들어 무작위 선택시 시나리오 작성시에 유형이나 장르를 랜덤으로 선택. |
| 장르 선택 버튼 | 유형 선택 후 해당 유형의 장르 목록 표시 |
| 고전 선택 시 추가 UI | 국가 선택 (한국 / 중국 / 일본) → 해당 국가의 장르 목록 표시, 국가 선택과 해당 국가의 장르 목록에도 무작위 버튼 추가 |

### 유형별 장르 목록

| 유형 | 선택 가능한 장르 |
|------|----------------|
| 영화 / 시리즈 / 만화 | 드라마 / 멜로·로맨스 / 스릴러 / 판타지 / 액션 / 미스터리 / 코미디 / SF / 전쟁 / 공포(호러) |
| 소설 | 드라마 / 멜로·로맨스 / 스릴러 / 판타지 / 액션 / 미스터리 / 코미디 / SF / 전쟁 |
| 고전 — 한국 | 가문소설 / 판타지 / 로맨스 / 영웅소설 / 미스터리 / 공포(호러) |
| 고전 — 중국 | 무협 / 로맨스 / 공포(호러) / 판타지 / 미스터리 |
| 고전 — 일본 | 설화 / 미스터리 / 공포(호러) |

> ⚠️ 소설 유형에서 공포(호러)를 표시하지 말 것 — 해당 데이터가 없음

### 동작

- 무작위 버튼 클릭 시: 백엔드에서 유형·장르 랜덤 선택 후 선택 결과 UI에 표시 X 결과에만 반영 → 자동으로 화면 2로 이동
- 유형·장르 직접 선택 완료 시: "다음" 버튼 → 화면 2로 이동
- 선택된 유형·장르는 세션 전체에서 유지

## 화면 2 — 소재 & 캐릭터 입력

### UI 구조 (3개 섹션)

#### 섹션 A — 소재 입력

| 요소 | 설명 |
|------|------|
| 텍스트 입력창 | 유저가 직접 소재 입력 (예: "마력을 흡수하는 저주받은 소녀와 그녀를 지키는 몰락 기사") |
| **AI에게 맡김** 토글 | 클릭 시 입력창 비활성화. `generate_query_auto()` 호출하여 소재 자동 생성 |
| 자동 생성 소재 미리보기 | AI에게 맡김 선택 시 생성된 소재 텍스트를 읽기 전용으로 표시 |

#### 섹션 B — AI 캐릭터 입력 (챗에서 AI가 연기할 캐릭터)

> **AI 뱃지를 누르면 해당 항목을 시나리오에 맞게 AI가 자동으로 설정해요.**
> 각 입력창 우측의 `AI` 뱃지를 개별적으로 선택할 수 있습니다.
> - 뱃지 비활성(회색): 유저가 직접 입력
> - 뱃지 활성(파란색): 입력창 비활성화 → AI가 시나리오 기반으로 자동 설정

| 입력 필드 | 설명 | AI 뱃지 |
|----------|------|---------|
| 이름 | 텍스트 입력 | 입력창 우측 `AI` 뱃지 버튼 |
| 성격 | 텍스트 입력 | 입력창 우측 `AI` 뱃지 버튼 |
| 외형 | 텍스트 입력 | 입력창 우측 `AI` 뱃지 버튼 |
| 배경 | 텍스트 입력 | 입력창 우측 `AI` 뱃지 버튼 |

- 모든 필드의 AI 뱃지를 한번에 활성화하는 **전체 AI에게 맡김** 버튼 제공
- 뱃지가 활성화된 필드는 `generate_characters()`가 시나리오 기반으로 자동 설계
- 뱃지가 비활성화된 필드(직접 입력값)는 AI 결과와 무관하게 유저 입력값으로 고정

#### 섹션 C — 유저 캐릭터 입력 (챗에서 유저가 연기할 캐릭터)

> **AI 뱃지를 누르면 해당 항목을 시나리오에 맞게 AI가 자동으로 설정해요.**
> 각 입력창 우측의 `AI` 뱃지를 개별적으로 선택할 수 있습니다.
> - 뱃지 비활성(회색): 유저가 직접 입력
> - 뱃지 활성(파란색): 입력창 비활성화 → AI가 시나리오 기반으로 자동 설정

| 입력 필드 | 설명 | AI 뱃지 |
|----------|------|---------|
| 이름 | 텍스트 입력 | 입력창 우측 `AI` 뱃지 버튼 |
| 성격 | 텍스트 입력 | 입력창 우측 `AI` 뱃지 버튼 |
| 외형 | 텍스트 입력 | 입력창 우측 `AI` 뱃지 버튼 |
| 배경 | 텍스트 입력 | 입력창 우측 `AI` 뱃지 버튼 |

- 모든 필드의 AI 뱃지를 한번에 활성화하는 **전체 AI에게 맡김** 버튼 제공
- 뱃지가 활성화된 필드는 `generate_characters()`가 시나리오 기반으로 자동 설계
- 뱃지가 비활성화된 필드(직접 입력값)는 AI 결과와 무관하게 유저 입력값으로 고정
- 이 섹션에서 입력한 유저 캐릭터 정보는 **화면 5 유저 페르소나에 자동으로 불러와집니다**

### 생성 시작

- **"시나리오 생성 시작"** 버튼 → 화면 3으로 이동
- 호출 함수: `generate_scenario_full(genre, category, user_query, ai_character, user_character)`


## 화면 3 — 생성 중 로딩

### UI 요소

| 요소 | 설명 |
|------|------|
| 단계별 진행 상태 텍스트 | 현재 어떤 작업 중인지 표시 |
| 로딩 애니메이션 | 진행 인디케이터 |

### 단계별 상태 메시지 순서

```
1. "소재 분석 중..."            → RAG 검색 (retrieve_scenes)
2. "시나리오 작성 중..."         → generate_scenario_with_rag() [Claude Sonnet — 10~20초 소요]
3. "캐릭터 설계 중..."           → generate_characters()
4. "인터랙티브 구조 변환 중..."   → build_scenario_tree() + expand_branch() x4
5. "엔딩 조건 설계 중..."        → generate_ending_conditions()
6. "완료!"                      → 화면 4로 자동 이동
```

> 시나리오 생성(Claude Sonnet) 단계가 가장 오래 걸립니다.

## 화면 4 — 완료 화면: 시나리오 & 캐릭터 확인

### UI 구조

#### 섹션 A — 배경 요약

- 생성된 기승전결 시나리오를 요약 형태로 표시
- **펼치기/접기** 버튼으로 전체 원문 확인 가능
- 기 / 승 / 전 / 결 단계별 탭으로 구분 표시

#### 섹션 B — 등장인물 목록

| 항목 | 표시 내용 | 수정 가능 |
|------|----------|----------|
| AI 캐릭터 카드 | 이름 / 역할 / 성격 / 외형 / 배경 / 이미지 | ✅ |
| 유저 캐릭터 카드 | 이름 / 역할 / 성격 / 배경 | ✅ |
| 조연 캐릭터 카드 (2~4명) | 이름 / 역할 / 배경 / 중요도 | ✅ |

**role 표시**: "냉혹한 용병", "복수를 꿈꾸는 몰락 귀족" 등 서사적 역할 자유 서술 형태로 표시
(주인공/조연 같은 고정 레이블 아님)

#### AI 캐릭터 이미지

- 완료 화면에서 AI 캐릭터 외형 설명 기반으로 **기준 이미지** 1장 생성
- 이후 챗 전 단계에서 이 이미지를 얼굴 레퍼런스로 재사용 (얼굴 일관성 유지)
- 이미지 생성 API: 미정 (추후 결정)

#### 수정 UI

- 각 캐릭터 카드에 **"수정"** 버튼
- 클릭 시 해당 필드 인라인 편집 또는 모달 팝업
- 수정 완료 후 **"확정"** 버튼

#### 플레이 시작

- **"플레이 시작"** 버튼 → 화면 5로 이동
- 이 시점에서 `GameState` 초기화 (affinity=0, current_node_id="기", turn_count=0)

## 화면 5 — 플레이 시작 전 설정

### UI 요소

#### 유저 페르소나

> 화면 2에서 입력한 유저 캐릭터 정보가 자동으로 불러와집니다.
> 그대로 플레이하거나, 필요한 경우 이 화면에서 자유롭게 수정할 수 있습니다.
> (화면 2에서 AI 뱃지로 맡긴 필드는 AI가 설정한 값으로 채워집니다.)

| 필드 | 자동 채워짐 출처 | 필수 여부 |
|------|----------------|---------|
| 이름 (챗에서 AI가 부를 이름) | 화면 2 유저 캐릭터 — 이름 | 권장 |
| 성격 | 화면 2 유저 캐릭터 — 성격 | 선택 |
| 직업 / 배경 | 화면 2 유저 캐릭터 — 배경 | 선택 |
| 외형(채팅 중 등장인물들이 인식하는 외형) | 화면 2 유저 캐릭터 — (AI 설계 시 자동 채워짐) | 선택 |

- 모든 필드는 유저가 자유롭게 수정 가능
- 화면 2에서 입력하지 않은 필드는 빈 상태로 표시 (선택적으로 직접 입력 가능)

#### 유저 노트

- AI가 **항상 기억할 사항**을 자유 텍스트로 입력
- 예: "나는 이 캐릭터를 무뚝뚝하게 연기하고 싶어"
- 매 챗 턴 시스템 프롬프트에 고정 주입 (빈번한 반영으로 출력 빈도 최소화)

#### 세션 옵션

| 옵션 | 선택지 |
|------|--------|
| 출력 모델 | Gemini Flash / Gemini Pro / gpt-4o-mini / gpt-4o |
| 추론 양 | 저 / 중 / 고 |
| 감성 | 긍정 / 중립 / 부정 |
| AI 주도 사건 | ON / OFF |
| 사칭 설정 | ON / OFF |
| 시작 설정 | 커스텀 / AI 추천 / 랜덤 |

### 시작 버튼

- **"대화 시작"** 버튼 → 화면 6으로 이동


## 화면 6 — 인터랙티브 챗 & 플레이

### 화면 레이아웃

```
┌─────────────────────────────────────────────────┐
│  [AI 캐릭터 이미지]  [캐릭터 이름]  [상태창]       │
│  ─────────────────────────────────────────────  │
│                                                 │
│  [대화 내용 영역]                                 │
│  AI: "..."                                      │
│  나: "..."                                      │
│                                                 │
│  ─────────────────────────────────────────────  │
│  [입력창]                           [전송 버튼]  │
│  [추천 대화 버튼 1] [버튼 2] [버튼 3]            │
└─────────────────────────────────────────────────┘
```

### 대화 영역

| 표시 유형 | 형식 |
|----------|------|
| AI 발화 | `"대화 내용"` 형식 |
| 유저 발화 | `"대화 내용"` 형식 |
| 서술 | 일반 텍스트 (행동·상황 묘사) |
| 분기점 이미지 | 분기 진입 시 AI 캐릭터 이미지 인라인 표시 |

### 입력창

| 입력 방식 | 동작 |
|----------|------|
| `"대화"` 형식 | AI 캐릭터에게 대화 |
| 서술 형식 | 유저 행동/상황 서술 |
| `!요약` 명령어 | 현재까지 대화 즉시 요약 후 저장 |
| `!설정` 명령어 | 추가 세계관 설정 입력 |

### 상태창

| 항목 | 설명 |
|------|------|
| 호감도 | -100 ~ 100 수치 + 바 (GameState.affinity) |
| 현재 단계 | 기 / 승 / 전 / 결 (GameState.current_node_id) |
| AI 캐릭터 내면 생각 | 매 턴 업데이트 |
| 대화 추천 | 3개 버튼 — 클릭 시 입력창에 자동 입력 |

### 분기점 도달 시 UI

```
┌────────────────────────────────────┐
│  선택의 순간                        │
│                                    │
│  [선택 A] 설명 텍스트               │
│  [선택 B] 설명 텍스트               │
└────────────────────────────────────┘
```

- 유저 선택 시: `GameState.record_choice(choice_id, affinity_delta)` 호출
- 해당 분기 시나리오(전_A 또는 전_B)로 대화 계속
- 분기 진입 시 AI 캐릭터 이미지 재생성 (기준 이미지 + 현재 상황 묘사)

### 이탈 감지 & 부분 재생성

- N턴 연속으로 시나리오 방향과 무관한 대화 감지 시 자동 트리거
- `regenerate_from_node(tree, current_node_id, conversation_summary, relationship_state, ...)` 호출
- 현재 위치 이후 노드만 재생성 (이미 진행된 부분은 유지)
- 재생성 중 로딩 표시 (백그라운드 처리 권장)

### 엔딩 체크 (매 턴)

- `check_ending_reached(game_state, tree)` 호출
- 결 단계 도달 시 자동으로 화면 7로 이동

## 화면 7 — 엔딩 결과 화면

### UI 요소

| 요소 | 설명 |
|------|------|
| 엔딩 타입 배지 | 해피 엔딩 / 배드 엔딩 / 트루 엔딩 / 루트 엔딩 |
| 엔딩 명칭 | 예: "진실의 화해", "결별 엔딩" |
| 엔딩 씬 텍스트 | LLM이 생성한 엔딩 장면 텍스트 |
| 엔딩 이미지 | 기준 이미지 + 엔딩 상황 묘사 (이미지 API 미정) |
| 달성 경로 표시 | 선택 A / B 분기 경로 시각화 |
| 다른 엔딩 힌트 | "다른 엔딩이 있습니다" 문구 (선택적) |
| 하단 버튼 | 공유 / 저장 / 재도전 |

### 엔딩 판정

- `evaluate_ending(game_state, ending_node)` 호출
- 우선순위: 트루 엔딩 override → forbidden_flags → affinity 범위 → 기본 타입
- 결과를 엔딩 타입 배지와 명칭으로 화면에 표시

## 서포트 레이어 — 로어북

### 자동 초기화 (화면 4 완료 시점)

- 시나리오 생성 완료 후 **자동으로** 핵심 항목 추출 → Vector DB 색인
- 추출 항목: 지명 / 인물 관계 / 고유 명사 / 역사적 사건 / 복선 디테일
- 유저 입력 불필요 (백그라운드 처리)

### 로어북 관리 UI (설정 메뉴 또는 사이드패널)

| 기능 | 설명 |
|------|------|
| 항목 목록 조회 | 자동 추출 항목 + 유저 추가 항목 표시 |
| 항목 수동 추가 | 유저가 직접 세계관 설정 등록 |
| 항목 수정 / 삭제 | 인라인 편집 |
| 항목 활성화 / 비활성화 | 토글로 특정 항목을 챗에서 제외 가능 |

### 챗 단계 동작 (백엔드 처리)

- 매 턴: 현재 대화 마지막 3~5턴 → Semantic Search → 문맥 유사 항목 시스템 프롬프트에 자동 주입
- 복선 관련 장면 감지 시 해당 항목 포함
- 과거 사건 ↔ 현재 대화 충돌 감지 (Narrative Consistency Engine)

## 계정 & 세션 관리

### 로그인

- 구글 로그인 (PC / 모바일 연동)

### 세션 목록 UI

| 기능 | 설명 |
|------|------|
| 세션 목록 | 진행 중 / 완료된 시나리오 목록 |
| 새 대화 생성 | + 버튼 → 화면 1부터 시작 |
| 세션 이름 수정 | 인라인 편집 |
| 세션 삭제 | 확인 다이얼로그 후 삭제 |
| 대화 수정 | 특정 턴 클릭 → 이후 내용 삭제 → 해당 시점부터 재진행 |
| 대화 공유 | 해당 시점부터 독립 세션 분기 생성 |

### 엔딩 아카이브

- 달성한 엔딩 목록 저장 및 조회
- 표시 항목: 엔딩 타입 / 명칭 / 달성 날짜 / 시나리오 제목

## 백엔드 함수 — 화면별 매핑

> 벡터DB.ipynb에 구현된 함수들을 API 엔드포인트로 래핑해야 합니다.

| 함수 | 파일 위치 | 호출 시점 |
|------|----------|----------|
| `generate_query_auto(genre, category)` | 벡터DB.ipynb §12 Cell 42 | 화면 2 — 소재 AI 자동 생성 |
| `generate_scenario_full(...)` | 벡터DB.ipynb §12 Cell 44 | 화면 2 — 생성 시작 버튼 |
| `generate_scenario_with_rag(...)` | 벡터DB.ipynb §10 Cell 28 | generate_scenario_full 내부 |
| `generate_characters(...)` | 벡터DB.ipynb §12 Cell 43 | generate_scenario_full 내부 |
| `build_scenario_tree(...)` | 벡터DB.ipynb §11 | 화면 3 — 트랜스포머 단계 |
| `expand_branch(...)` | 벡터DB.ipynb §11 Cell 31 | build_scenario_tree 내부 |
| `expand_linear_node(...)` | 벡터DB.ipynb §11 Cell 34 | regenerate_from_node 내부 |
| `regenerate_from_node(...)` | 벡터DB.ipynb §11 Cell 35 | 화면 6 — 이탈 감지 시 |
| `generate_ending_conditions(...)` | 벡터DB.ipynb §11 Cell 38 | 화면 3 — 엔딩 조건 설계 단계 |
| `evaluate_ending(game_state, node)` | 벡터DB.ipynb §11 Cell 39 | 화면 6 — 매 턴 엔딩 체크 |
| `check_ending_reached(game_state, tree)` | 벡터DB.ipynb §11 Cell 39 | 화면 6 — 매 턴 자동 체크 |
| `retrieve_scenes(...)` | 벡터DB.ipynb §7 | generate_scenario_with_rag 내부 |
| `build_rag_context(...)` | 벡터DB.ipynb §8 | generate_scenario_with_rag 내부 |

### GameState 관리

```python
# GameState 필드 (벡터DB.ipynb §11 Cell 37)
GameState(
    affinity=0,               # 호감도: -100 ~ 100
    choice_history=[],        # 선택 이력 ["A", "B", ...]
    event_flags=set(),        # 달성된 이벤트 플래그
    current_node_id="기",     # 현재 노드: 기/승/전/전_A/전_B/결_A/결_B
    turn_count=0              # 총 턴 수
)
```

- 세션 단위로 서버에서 유지 (또는 DB에 직렬화)
- `GameState.to_dict()` / `from_dict()` 로 직렬화/역직렬화 가능
- 매 챗 턴 후 DB 저장 권장

### generate_characters 반환 구조 (캐릭터 데이터)

```json
{
    "ai_character": {
        "name": "캐릭터 이름",
        "role": "서사적 역할 (자유 서술 — 예: 복수를 꿈꾸는 몰락 귀족)",
        "personality": "성격",
        "appearance": "외형",
        "background": "배경",
        "relationship_to_user": "유저 캐릭터와의 관계",
        "context": "챗 시스템 프롬프트에 주입할 자연어 설명 1~2문장"
    },
    "user_character": {
        "name": "캐릭터 이름",
        "role": "서사적 역할 (자유 서술)",
        "personality": "성격",
        "background": "배경",
        "context": "챗 시스템 프롬프트에 주입할 자연어 설명 1~2문장"
    },
    "supporting_characters": [
        {
            "name": "조연 이름",
            "role": "서사적 역할 (자유 서술)",
            "personality": "성격",
            "background": "배경",
            "relationship": "두 주인공과의 관계",
            "importance": "주요 또는 보조"
        }
    ]
}
```

> `context` 필드: 챗 시스템 프롬프트에 그대로 주입. AI가 해당 캐릭터를 어떻게 연기할지 자연어로 설명.

## 미구현 / 미정 항목

| 항목 | 상태 | 비고 |
|------|------|------|
| AI 캐릭터 이미지 생성 | ⏳ API 미정 | Leonardo.ai Character Reference / Civitai IP-Adapter 검토 중 |
| 얼굴 일관성 유지 방식 | ⏳ 미정 | Character Reference 방식 검토 중 |
| 챗 출력 모델 | ⏳ 미정 | Gemini Flash / Pro 검토 중 |
| 로어북 자동 초기화 | 🔧 미구현 | 시나리오 생성 후 핵심 항목 자동 추출 함수 필요 |
| 인터랙티브 챗 시스템 프롬프트 | 🔧 미구현 | 로어북 + RAG + 요약기억 동적 조합 로직 |
| 이탈 감지 로직 | 🔧 미구현 | N턴 기준 정의 및 감지 함수 |
| 대화 요약기 | 🔧 미구현 | 10~15턴 초과 시 자동 요약 |
| 엔딩 씬 생성 | 🔧 미구현 | 엔딩 조건 달성 시 LLM으로 엔딩 장면 텍스트 생성 |